# Principal Component Analysis (PCA) - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Principal Component Analysis (PCA)?

PCA is a **dimensionality reduction** technique that transforms high-dimensional data into a lower-dimensional space while preserving as much **variance** (information) as possible. It identifies the directions (principal components) along which the data varies the most.

---

### 1.1 Variance Maximization Perspective

The goal of PCA is to find a set of orthogonal axes (principal components) that maximize the variance of the projected data.

**Objective:** Find unit vector $\mathbf{u}_1$ that maximizes the variance of the projection:

$$\max_{\mathbf{u}_1} \text{Var}(\mathbf{X} \mathbf{u}_1) = \max_{\mathbf{u}_1} \mathbf{u}_1^T \mathbf{S} \mathbf{u}_1$$

Subject to: $\mathbf{u}_1^T \mathbf{u}_1 = 1$ (unit vector constraint)

Where $\mathbf{S}$ is the covariance matrix of the data.

---

### 1.2 Covariance Matrix

For centered data $\mathbf{X}$ (mean-subtracted), the covariance matrix is:

$$\mathbf{S} = \frac{1}{n-1} \mathbf{X}^T \mathbf{X}$$

Where:
- $\mathbf{X}$ is an $(n \times d)$ matrix (n samples, d features)
- $\mathbf{S}$ is a $(d \times d)$ symmetric positive semi-definite matrix

The diagonal elements represent variances of individual features, and off-diagonal elements represent covariances between feature pairs.

---

### 1.3 Eigendecomposition Approach

Using Lagrange multipliers to solve the constrained optimization:

$$\mathcal{L} = \mathbf{u}_1^T \mathbf{S} \mathbf{u}_1 - \lambda(\mathbf{u}_1^T \mathbf{u}_1 - 1)$$

Taking the derivative and setting to zero:

$$\mathbf{S} \mathbf{u}_1 = \lambda \mathbf{u}_1$$

This shows that the principal components are the **eigenvectors** of the covariance matrix, and the eigenvalues represent the variance along each component.

**Key Result:** The variance of data projected onto eigenvector $\mathbf{u}_i$ equals the eigenvalue $\lambda_i$:
$$\text{Var}(\mathbf{X} \mathbf{u}_i) = \lambda_i$$

---

### 1.4 Singular Value Decomposition (SVD) Approach

An alternative (and numerically more stable) approach uses SVD:

$$\mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$$

Where:
- $\mathbf{U}$ is $(n \times n)$ orthogonal matrix (left singular vectors)
- $\mathbf{\Sigma}$ is $(n \times d)$ diagonal matrix (singular values)
- $\mathbf{V}$ is $(d \times d)$ orthogonal matrix (right singular vectors)

**Connection to Eigendecomposition:**
- The columns of $\mathbf{V}$ are the principal components (eigenvectors of $\mathbf{X}^T\mathbf{X}$)
- The eigenvalues of the covariance matrix: $\lambda_i = \frac{\sigma_i^2}{n-1}$

**Advantages of SVD:**
1. Numerically more stable (avoids computing $\mathbf{X}^T\mathbf{X}$)
2. More efficient for $n < d$ (tall matrices)
3. Works directly on the data matrix

---

### 1.5 Explained Variance Ratio

The explained variance ratio for the $k$-th component is:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_{i=1}^{d} \lambda_i}$$

The cumulative explained variance for the first $k$ components:

$$\text{Cumulative EVR}_k = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{i=1}^{d} \lambda_i}$$

---

### 1.6 Choosing n_components

Several strategies exist:

1. **Cumulative Variance Threshold:** Select $k$ such that cumulative variance $\geq$ threshold (e.g., 95%)

2. **Elbow Method:** Look for an "elbow" in the scree plot where variance drops significantly

3. **Kaiser Criterion:** Keep components with eigenvalues $> 1$ (for standardized data)

4. **Cross-Validation:** Evaluate downstream task performance for different $k$

---

### 1.7 Time and Space Complexity

| Operation | Eigendecomposition | SVD (Full) | SVD (Truncated) |
|-----------|-------------------|------------|------------------|
| **Time** | $O(d^3 + nd^2)$ | $O(\min(nd^2, n^2d))$ | $O(ndk)$ |
| **Space** | $O(d^2)$ | $O(\min(n,d)^2)$ | $O((n+d)k)$ |

Where $n$ = samples, $d$ = features, $k$ = components

**Practical Notes:**
- For $d > n$: SVD on $\mathbf{X}$ is more efficient than eigendecomposition of $\mathbf{X}^T\mathbf{X}$
- For very large datasets: Use randomized SVD or incremental PCA
- Memory bottleneck: Covariance matrix requires $O(d^2)$ storage

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class PCA:
    """
    Principal Component Analysis (PCA) implementation from scratch.
    
    This implementation supports both eigendecomposition and SVD methods
    for computing principal components.
    
    Parameters
    ----------
    n_components : int or float or None, default=None
        Number of components to keep.
        - If int: exact number of components
        - If float (0 < n_components < 1): select components to explain
          at least this fraction of variance
        - If None: keep all components
    
    method : str, default='svd'
        Method to compute principal components.
        - 'svd': Singular Value Decomposition (recommended)
        - 'eigen': Eigendecomposition of covariance matrix
    
    Attributes
    ----------
    components_ : ndarray of shape (n_components, n_features)
        Principal axes in feature space (eigenvectors)
    
    explained_variance_ : ndarray of shape (n_components,)
        Variance explained by each component (eigenvalues)
    
    explained_variance_ratio_ : ndarray of shape (n_components,)
        Percentage of variance explained by each component
    
    mean_ : ndarray of shape (n_features,)
        Per-feature mean computed from training data
    
    n_components_ : int
        Number of components kept after fitting
    
    n_features_ : int
        Number of features in the training data
    
    n_samples_ : int
        Number of samples in the training data
    """
    
    def __init__(self, n_components=None, method='svd'):
        self.n_components = n_components
        self.method = method
        
        # Attributes set during fit
        self.components_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None
        self.n_components_ = None
        self.n_features_ = None
        self.n_samples_ = None
        self._singular_values = None  # For reconstruction
    
    def _validate_n_components(self, n_features):
        """
        Validate and process n_components parameter.
        """
        if self.n_components is None:
            return n_features
        elif isinstance(self.n_components, float):
            if not 0 < self.n_components < 1:
                raise ValueError(
                    "n_components as float must be in (0, 1), "
                    f"got {self.n_components}"
                )
            # Will be determined after computing explained variance
            return None
        elif isinstance(self.n_components, int):
            if self.n_components <= 0:
                raise ValueError(
                    f"n_components must be positive, got {self.n_components}"
                )
            if self.n_components > n_features:
                raise ValueError(
                    f"n_components ({self.n_components}) cannot exceed "
                    f"n_features ({n_features})"
                )
            return self.n_components
        else:
            raise TypeError(
                f"n_components must be int, float, or None, "
                f"got {type(self.n_components)}"
            )
    
    def _select_n_components_by_variance(self, explained_variance_ratio):
        """
        Select number of components to explain desired variance fraction.
        """
        cumulative_variance = np.cumsum(explained_variance_ratio)
        n_components = np.searchsorted(cumulative_variance, self.n_components) + 1
        return min(n_components, len(explained_variance_ratio))
    
    def _fit_svd(self, X):
        """
        Fit PCA using Singular Value Decomposition.
        
        This is the numerically stable approach that avoids
        explicitly computing the covariance matrix.
        """
        n_samples, n_features = X.shape
        
        # Center the data
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Perform SVD
        # X = U * S * V^T
        # For efficiency, use full_matrices=False
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Principal components are rows of Vt (columns of V)
        # V^T has shape (min(n,d), d)
        components = Vt
        
        # Explained variance from singular values
        # Var = S^2 / (n - 1)
        explained_variance = (S ** 2) / (n_samples - 1)
        total_variance = np.sum(explained_variance)
        explained_variance_ratio = explained_variance / total_variance
        
        # Store singular values for potential reconstruction
        self._singular_values = S
        
        return components, explained_variance, explained_variance_ratio
    
    def _fit_eigen(self, X):
        """
        Fit PCA using eigendecomposition of the covariance matrix.
        
        Note: This approach is less numerically stable than SVD,
        especially for ill-conditioned data.
        """
        n_samples, n_features = X.shape
        
        # Center the data
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Compute covariance matrix
        # Using n-1 for unbiased estimate (Bessel's correction)
        cov_matrix = np.dot(X_centered.T, X_centered) / (n_samples - 1)
        
        # Eigendecomposition
        # Returns eigenvalues in ascending order
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Sort in descending order
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Principal components are columns of eigenvectors, transpose to rows
        components = eigenvectors.T
        
        # Explained variance
        explained_variance = eigenvalues
        # Handle potential numerical issues (small negative eigenvalues)
        explained_variance = np.maximum(explained_variance, 0)
        total_variance = np.sum(explained_variance)
        explained_variance_ratio = explained_variance / total_variance
        
        return components, explained_variance, explained_variance_ratio
    
    def fit(self, X):
        """
        Fit the PCA model to the data.
        
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        
        Returns
        -------
        self : PCA
            Fitted PCA instance.
        """
        # Convert to numpy array
        X = np.asarray(X, dtype=np.float64)
        
        if X.ndim != 2:
            raise ValueError(f"Expected 2D array, got {X.ndim}D array instead")
        
        self.n_samples_, self.n_features_ = X.shape
        
        # Fit using specified method
        if self.method == 'svd':
            components, explained_variance, explained_variance_ratio = self._fit_svd(X)
        elif self.method == 'eigen':
            components, explained_variance, explained_variance_ratio = self._fit_eigen(X)
        else:
            raise ValueError(f"Unknown method '{self.method}'. Use 'svd' or 'eigen'.")
        
        # Determine number of components
        n_components = self._validate_n_components(self.n_features_)
        if n_components is None:  # Float case
            n_components = self._select_n_components_by_variance(explained_variance_ratio)
        
        # Keep only the requested number of components
        self.n_components_ = n_components
        self.components_ = components[:n_components]
        self.explained_variance_ = explained_variance[:n_components]
        self.explained_variance_ratio_ = explained_variance_ratio[:n_components]
        
        return self
    
    def transform(self, X):
        """
        Apply dimensionality reduction to X.
        
        Projects X onto the principal components.
        
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Data to transform.
        
        Returns
        -------
        X_transformed : ndarray of shape (n_samples, n_components)
            Transformed data in the new coordinate system.
        """
        if self.components_ is None:
            raise RuntimeError("PCA must be fitted before transform. Call fit() first.")
        
        X = np.asarray(X, dtype=np.float64)
        
        if X.shape[1] != self.n_features_:
            raise ValueError(
                f"X has {X.shape[1]} features, but PCA was fitted with {self.n_features_} features"
            )
        
        # Center the data using the mean from training
        X_centered = X - self.mean_
        
        # Project onto principal components
        # X_transformed = X_centered @ components.T
        X_transformed = np.dot(X_centered, self.components_.T)
        
        return X_transformed
    
    def fit_transform(self, X):
        """
        Fit the model and apply dimensionality reduction to X.
        
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        
        Returns
        -------
        X_transformed : ndarray of shape (n_samples, n_components)
            Transformed data.
        """
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, X_transformed):
        """
        Transform data back to original space.
        
        Note: If n_components < n_features, this is a lossy reconstruction.
        
        Parameters
        ----------
        X_transformed : array-like of shape (n_samples, n_components)
            Data in the transformed space.
        
        Returns
        -------
        X_reconstructed : ndarray of shape (n_samples, n_features)
            Reconstructed data in original space.
        """
        if self.components_ is None:
            raise RuntimeError("PCA must be fitted before inverse_transform. Call fit() first.")
        
        X_transformed = np.asarray(X_transformed, dtype=np.float64)
        
        if X_transformed.shape[1] != self.n_components_:
            raise ValueError(
                f"X_transformed has {X_transformed.shape[1]} components, "
                f"but PCA has {self.n_components_} components"
            )
        
        # Reconstruct: X_reconstructed = X_transformed @ components + mean
        X_reconstructed = np.dot(X_transformed, self.components_) + self.mean_
        
        return X_reconstructed
    
    def get_covariance(self):
        """
        Compute the covariance matrix from the fitted components.
        
        Returns
        -------
        cov : ndarray of shape (n_features, n_features)
            Estimated covariance matrix.
        """
        if self.components_ is None:
            raise RuntimeError("PCA must be fitted first.")
        
        # Cov = V @ diag(eigenvalues) @ V^T
        return np.dot(
            self.components_.T * self.explained_variance_,
            self.components_
        )

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Load the digits dataset
digits = load_digits()
X = digits.data
y = digits.target

print("Digits Dataset Information:")
print("="*50)
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Feature shape per sample: {digits.images[0].shape} (8x8 pixels)")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Classes: {np.unique(y)}")
print(f"\nFeature value range: [{X.min():.1f}, {X.max():.1f}]")

In [ ]:
# Visualize some sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits from Dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Standardize the data (important for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("After Standardization:")
print(f"Mean of features: {X_scaled.mean(axis=0).mean():.6f} (should be ~0)")
print(f"Std of features: {X_scaled.std(axis=0).mean():.6f} (should be ~1)")

In [ ]:
# Fit PCA with all components first to analyze variance
pca_full = PCA(n_components=None, method='svd')
pca_full.fit(X_scaled)

print("PCA Fitting Results (All Components):")
print("="*50)
print(f"Number of components: {pca_full.n_components_}")
print(f"Total variance explained: {sum(pca_full.explained_variance_ratio_):.4f}")
print(f"\nTop 10 components explained variance ratio:")
for i in range(10):
    print(f"  PC{i+1}: {pca_full.explained_variance_ratio_[i]:.4f} "
          f"({pca_full.explained_variance_ratio_[i]*100:.2f}%)")

In [ ]:
# Analyze cumulative explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components for different variance thresholds
thresholds = [0.80, 0.90, 0.95, 0.99]
print("\nComponents needed for different variance thresholds:")
print("-" * 40)
for threshold in thresholds:
    n_comp = np.searchsorted(cumulative_variance, threshold) + 1
    print(f"{threshold*100:.0f}% variance: {n_comp} components "
          f"(actual: {cumulative_variance[n_comp-1]*100:.2f}%)")

In [ ]:
# Fit PCA with 95% variance threshold
pca_95 = PCA(n_components=0.95, method='svd')
X_pca_95 = pca_95.fit_transform(X_scaled)

print("\nPCA with 95% Variance Threshold:")
print("="*50)
print(f"Original dimensions: {X_scaled.shape[1]}")
print(f"Reduced dimensions: {X_pca_95.shape[1]}")
print(f"Dimensionality reduction: {(1 - X_pca_95.shape[1]/X_scaled.shape[1])*100:.1f}%")
print(f"Actual variance explained: {sum(pca_95.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Compare SVD and Eigendecomposition methods
import time

# Time SVD method
start_svd = time.time()
pca_svd = PCA(n_components=20, method='svd')
X_svd = pca_svd.fit_transform(X_scaled)
time_svd = time.time() - start_svd

# Time Eigendecomposition method
start_eigen = time.time()
pca_eigen = PCA(n_components=20, method='eigen')
X_eigen = pca_eigen.fit_transform(X_scaled)
time_eigen = time.time() - start_eigen

print("\nMethod Comparison (20 components):")
print("="*50)
print(f"SVD method time: {time_svd*1000:.2f} ms")
print(f"Eigendecomposition time: {time_eigen*1000:.2f} ms")
print(f"\nExplained variance ratio agreement:")
print(f"Max absolute difference: {np.max(np.abs(pca_svd.explained_variance_ratio_ - pca_eigen.explained_variance_ratio_)):.2e}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def compute_reconstruction_error(pca, X):
    """
    Compute reconstruction error metrics.
    
    Parameters
    ----------
    pca : PCA
        Fitted PCA model
    X : ndarray
        Original data
    
    Returns
    -------
    dict : Dictionary containing various error metrics
    """
    # Transform and reconstruct
    X_transformed = pca.transform(X)
    X_reconstructed = pca.inverse_transform(X_transformed)
    
    # Compute errors
    errors = X - X_reconstructed
    
    # Mean Squared Error
    mse = np.mean(errors ** 2)
    
    # Root Mean Squared Error
    rmse = np.sqrt(mse)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(errors))
    
    # Relative Error (Frobenius norm)
    relative_error = np.linalg.norm(errors, 'fro') / np.linalg.norm(X, 'fro')
    
    # Per-sample reconstruction error
    per_sample_error = np.mean(errors ** 2, axis=1)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'relative_error': relative_error,
        'per_sample_error': per_sample_error
    }

In [ ]:
# Analyze reconstruction error for different number of components
n_components_range = [2, 5, 10, 20, 30, 40, 50, 64]
errors_by_components = []

print("Reconstruction Error Analysis:")
print("="*70)
print(f"{'n_components':>12} | {'MSE':>10} | {'RMSE':>10} | {'Relative Error':>15} | {'Var Explained':>13}")
print("-"*70)

for n_comp in n_components_range:
    pca_temp = PCA(n_components=n_comp, method='svd')
    pca_temp.fit(X_scaled)
    
    errors = compute_reconstruction_error(pca_temp, X_scaled)
    var_explained = sum(pca_temp.explained_variance_ratio_)
    
    errors_by_components.append({
        'n_components': n_comp,
        **errors,
        'var_explained': var_explained
    })
    
    print(f"{n_comp:>12} | {errors['mse']:>10.4f} | {errors['rmse']:>10.4f} | "
          f"{errors['relative_error']:>15.4f} | {var_explained*100:>12.2f}%")

In [ ]:
# Visualize explained variance ratio and cumulative variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance ratio
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1), 
            pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Explained Variance Ratio by Component')
axes[0].set_xlim(0, 65)

# Cumulative explained variance
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumulative) + 1), cumulative, 'o-', 
             color='steelblue', linewidth=2, markersize=4)
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.80, color='green', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend(loc='lower right')
axes[1].set_xlim(0, 65)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Reconstruction error vs variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_comps = [e['n_components'] for e in errors_by_components]
mses = [e['mse'] for e in errors_by_components]
rel_errors = [e['relative_error'] for e in errors_by_components]
var_exp = [e['var_explained'] for e in errors_by_components]

# MSE vs n_components
axes[0].plot(n_comps, mses, 'o-', color='crimson', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Mean Squared Error')
axes[0].set_title('Reconstruction Error vs Components')
axes[0].grid(True, alpha=0.3)

# Variance explained vs Relative error
axes[1].plot(var_exp, rel_errors, 'o-', color='forestgreen', linewidth=2, markersize=8)
axes[1].set_xlabel('Variance Explained')
axes[1].set_ylabel('Relative Reconstruction Error')
axes[1].set_title('Reconstruction Error vs Variance Explained')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze reconstruction quality for individual samples
pca_30 = PCA(n_components=30, method='svd')
pca_30.fit(X_scaled)
errors_30 = compute_reconstruction_error(pca_30, X_scaled)

# Per-sample error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of per-sample errors
axes[0].hist(errors_30['per_sample_error'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(np.mean(errors_30['per_sample_error']), color='red', linestyle='--', 
                label=f"Mean: {np.mean(errors_30['per_sample_error']):.4f}")
axes[0].axvline(np.median(errors_30['per_sample_error']), color='orange', linestyle='--', 
                label=f"Median: {np.median(errors_30['per_sample_error']):.4f}")
axes[0].set_xlabel('Per-Sample MSE')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Per-Sample Reconstruction Errors (30 components)')
axes[0].legend()

# Error by digit class
error_by_class = [errors_30['per_sample_error'][y == i] for i in range(10)]
axes[1].boxplot(error_by_class, labels=range(10))
axes[1].set_xlabel('Digit Class')
axes[1].set_ylabel('Reconstruction MSE')
axes[1].set_title('Reconstruction Error by Digit Class')

plt.tight_layout()
plt.show()

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# Scree Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Plot individual variance
n_show = 30
x = range(1, n_show + 1)
ax.bar(x, pca_full.explained_variance_[:n_show], alpha=0.6, color='steelblue', 
       label='Individual Variance')

# Plot cumulative on secondary axis
ax2 = ax.twinx()
cumsum = np.cumsum(pca_full.explained_variance_ratio_[:n_show])
ax2.plot(x, cumsum, 'ro-', linewidth=2, markersize=6, label='Cumulative Ratio')

# Add threshold lines
ax2.axhline(y=0.90, color='orange', linestyle='--', alpha=0.7)
ax2.axhline(y=0.95, color='red', linestyle='--', alpha=0.7)

ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Eigenvalue (Variance)', fontsize=12, color='steelblue')
ax2.set_ylabel('Cumulative Variance Ratio', fontsize=12, color='red')
ax.set_title('Scree Plot with Cumulative Variance', fontsize=14)

# Legends
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.show()

In [ ]:
# 2D Projection of digits data
pca_2d = PCA(n_components=2, method='svd')
X_2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 10))

# Create scatter plot with colors for each digit
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', 
                     alpha=0.7, s=30, edgecolor='none')

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label('Digit Class', fontsize=12)

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=12)
ax.set_title('2D PCA Projection of Digits Dataset', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Total variance explained by 2 components: "
      f"{sum(pca_2d.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# 3D Projection of digits data
pca_3d = PCA(n_components=3, method='svd')
X_3d = pca_3d.fit_transform(X_scaled)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Create 3D scatter plot
scatter = ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], 
                     c=y, cmap='tab10', alpha=0.7, s=20)

ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]*100:.2f}%)', fontsize=10)
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]*100:.2f}%)', fontsize=10)
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]*100:.2f}%)', fontsize=10)
ax.set_title('3D PCA Projection of Digits Dataset', fontsize=14)

# Add colorbar
cbar = fig.colorbar(scatter, ax=ax, shrink=0.6, pad=0.1)
cbar.set_label('Digit Class', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Total variance explained by 3 components: "
      f"{sum(pca_3d.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# Visualize principal components as images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    # Get the i-th principal component
    pc = pca_full.components_[i].reshape(8, 8)
    
    # Plot with diverging colormap
    im = ax.imshow(pc, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    ax.set_title(f'PC{i+1}\n({pca_full.explained_variance_ratio_[i]*100:.1f}%)', fontsize=10)
    ax.axis('off')

# Add colorbar
fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.8, label='Component Weight')
plt.suptitle('First 10 Principal Components as 8x8 Images', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize reconstruction quality
def visualize_reconstruction(X_original, pca_model, scaler, n_samples=5, sample_indices=None):
    """
    Visualize original vs reconstructed images.
    """
    if sample_indices is None:
        sample_indices = np.random.choice(len(X_original), n_samples, replace=False)
    
    # Get original images (unscaled)
    X_samples = X_original[sample_indices]
    
    # Scale, transform, inverse transform, unscale
    X_scaled_samples = scaler.transform(X_samples)
    X_transformed = pca_model.transform(X_scaled_samples)
    X_reconstructed_scaled = pca_model.inverse_transform(X_transformed)
    X_reconstructed = scaler.inverse_transform(X_reconstructed_scaled)
    
    # Create figure
    fig, axes = plt.subplots(3, n_samples, figsize=(3*n_samples, 9))
    
    for i, idx in enumerate(sample_indices):
        # Original
        axes[0, i].imshow(X_samples[i].reshape(8, 8), cmap='gray')
        axes[0, i].set_title(f'Original\nLabel: {y[idx]}')
        axes[0, i].axis('off')
        
        # Reconstructed
        axes[1, i].imshow(X_reconstructed[i].reshape(8, 8), cmap='gray')
        axes[1, i].set_title(f'Reconstructed\n({pca_model.n_components_} PCs)')
        axes[1, i].axis('off')
        
        # Difference
        diff = X_samples[i] - X_reconstructed[i]
        axes[2, i].imshow(diff.reshape(8, 8), cmap='RdBu_r', vmin=-5, vmax=5)
        mse = np.mean(diff**2)
        axes[2, i].set_title(f'Difference\nMSE: {mse:.2f}')
        axes[2, i].axis('off')
    
    axes[0, 0].set_ylabel('Original', fontsize=12)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=12)
    axes[2, 0].set_ylabel('Difference', fontsize=12)
    
    plt.tight_layout()
    plt.show()

# Compare reconstructions with different numbers of components
sample_indices = [0, 100, 500, 1000, 1500]

for n_comp in [2, 10, 30]:
    pca_temp = PCA(n_components=n_comp, method='svd')
    pca_temp.fit(X_scaled)
    print(f"\nReconstruction with {n_comp} components "
          f"({sum(pca_temp.explained_variance_ratio_)*100:.1f}% variance):")
    visualize_reconstruction(X, pca_temp, scaler, n_samples=5, sample_indices=sample_indices)

In [ ]:
# Biplot: Visualize samples and features together
def create_biplot(pca_model, X_transformed, feature_names=None, y=None, 
                  scale_factor=5, figsize=(12, 10)):
    """
    Create a biplot showing both samples and feature loadings.
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot samples
    if y is not None:
        scatter = ax.scatter(X_transformed[:, 0], X_transformed[:, 1], 
                            c=y, cmap='tab10', alpha=0.5, s=30)
        plt.colorbar(scatter, ax=ax, label='Class')
    else:
        ax.scatter(X_transformed[:, 0], X_transformed[:, 1], alpha=0.5, s=30)
    
    # Plot feature loadings (principal component vectors)
    loadings = pca_model.components_.T * np.sqrt(pca_model.explained_variance_) * scale_factor
    
    # Only show a subset of features to avoid clutter
    n_features = loadings.shape[0]
    if n_features > 20:
        # Show features with highest loadings
        loading_magnitudes = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
        top_features = np.argsort(loading_magnitudes)[-15:]
    else:
        top_features = range(n_features)
    
    for i in top_features:
        ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], 
                 head_width=0.2, head_length=0.1, fc='red', ec='red', alpha=0.7)
        label = feature_names[i] if feature_names is not None else f'F{i}'
        ax.text(loadings[i, 0]*1.1, loadings[i, 1]*1.1, label, 
                fontsize=8, color='red', alpha=0.8)
    
    ax.set_xlabel(f'PC1 ({pca_model.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_model.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA Biplot')
    ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

# Create feature names (pixel positions)
feature_names = [f'P{i//8},{i%8}' for i in range(64)]

create_biplot(pca_2d, X_2d, feature_names=feature_names, y=y, scale_factor=3)

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use PCA

#### Preprocessing for Machine Learning
- **Reduce dimensionality** before training models to avoid curse of dimensionality
- **Speed up training** by reducing feature count
- **Reduce storage** requirements for large datasets

#### Data Visualization
- Project high-dimensional data to 2D/3D for visualization
- Explore data structure and clustering patterns
- Identify outliers in reduced space

#### Noise Reduction
- Remove noise by keeping only top components (signal typically in high-variance directions)
- Image compression and denoising
- Sensor data smoothing

#### Feature Extraction
- Create uncorrelated features from correlated inputs
- Reduce multicollinearity in regression models
- Generate features for further analysis

---

### When NOT to Use PCA

#### Non-linear Relationships
PCA captures **linear** relationships only. For non-linear data:
- Use **Kernel PCA** for polynomial/RBF transformations
- Use **t-SNE** or **UMAP** for visualization
- Use **Autoencoders** for non-linear dimensionality reduction

#### When Interpretability is Crucial
- Principal components are **linear combinations** of original features
- Hard to interpret what each PC "means"
- Consider **feature selection** methods instead (Lasso, tree-based importance)

#### Different Scales Without Standardization
- PCA maximizes variance; features with larger scales dominate
- **Always standardize** before PCA unless scales are intentionally meaningful

#### Sparse Data
- PCA doesn't preserve sparsity
- Consider **Sparse PCA** or **NMF** for sparse representations

#### When Labels Matter
- PCA is **unsupervised**; ignores class information
- For supervised dimensionality reduction, use **LDA** (Linear Discriminant Analysis)

---

### Component Selection Strategies

| Strategy | Description | When to Use |
|----------|-------------|-------------|
| **Cumulative Variance** | Keep components until 90-99% variance | General purpose, safe choice |
| **Elbow Method** | Visual inspection of scree plot | When clear "elbow" exists |
| **Kaiser Criterion** | Keep eigenvalues > 1 (standardized data) | Quick heuristic |
| **Cross-Validation** | Evaluate downstream task performance | When specific task is known |
| **Parallel Analysis** | Compare with random data eigenvalues | Statistical rigor |

---

### Importance of Feature Scaling

**Critical:** PCA is sensitive to feature scales because it maximizes variance.

```
Example:
Feature A: Income in dollars (range: 20000-200000)
Feature B: Age in years (range: 18-80)

Without scaling: PC1 dominated by Income (higher variance due to scale)
With scaling: Both features contribute equally based on their relative variability
```

**Recommendation:**
- Use **StandardScaler** (z-score normalization) before PCA
- Exception: When absolute scale carries meaning (e.g., all features already in same units)

In [ ]:
# Demonstration: Impact of scaling on PCA
# Create synthetic data with different scales
np.random.seed(42)
n_samples = 500

# Feature 1: Large scale (thousands)
feature1 = np.random.normal(50000, 10000, n_samples)
# Feature 2: Small scale (0-1)
feature2 = np.random.normal(0.5, 0.1, n_samples)
# Feature 3: Medium scale (hundreds)
feature3 = np.random.normal(100, 20, n_samples)

X_demo = np.column_stack([feature1, feature2, feature3])

# PCA without scaling
pca_no_scale = PCA(n_components=3)
pca_no_scale.fit(X_demo)

# PCA with scaling
X_demo_scaled = StandardScaler().fit_transform(X_demo)
pca_scaled = PCA(n_components=3)
pca_scaled.fit(X_demo_scaled)

print("Impact of Scaling on PCA:")
print("="*60)
print("\nWithout Scaling:")
print(f"  PC1 loadings: {pca_no_scale.components_[0]}")
print(f"  Explained variance ratio: {pca_no_scale.explained_variance_ratio_}")
print("\nWith Scaling:")
print(f"  PC1 loadings: {pca_scaled.components_[0]}")
print(f"  Explained variance ratio: {pca_scaled.explained_variance_ratio_}")

print("\nObservation: Without scaling, Feature 1 (large scale) dominates PC1.")
print("With scaling, all features contribute more equally.")

In [ ]:
# Demonstration: PCA fails on non-linear data
# Create circular/spiral data
np.random.seed(42)
n_points = 1000

# Create 2D spiral
t = np.linspace(0, 4*np.pi, n_points)
noise = np.random.normal(0, 0.1, n_points)
x_spiral = t * np.cos(t) + noise
y_spiral = t * np.sin(t) + noise
X_spiral = np.column_stack([x_spiral, y_spiral])
colors_spiral = t

# Apply PCA
pca_spiral = PCA(n_components=2)
X_spiral_pca = pca_spiral.fit_transform(X_spiral)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original spiral
scatter1 = axes[0].scatter(X_spiral[:, 0], X_spiral[:, 1], c=colors_spiral, cmap='viridis', s=20)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Original Spiral Data')
axes[0].set_aspect('equal')
plt.colorbar(scatter1, ax=axes[0], label='Position along spiral')

# PCA projection (just shows principal axes, doesn't unroll)
scatter2 = axes[1].scatter(X_spiral_pca[:, 0], X_spiral_pca[:, 1], c=colors_spiral, cmap='viridis', s=20)
axes[1].set_xlabel(f'PC1 ({pca_spiral.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_spiral.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA Cannot Unroll Non-linear Structure')
plt.colorbar(scatter2, ax=axes[1], label='Position along spiral')

plt.tight_layout()
plt.show()

print("Note: PCA finds linear directions of maximum variance.")
print("It cannot 'unroll' the spiral - for that, use t-SNE, UMAP, or Kernel PCA.")

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn's PCA
from sklearn.decomposition import PCA as SklearnPCA

# Test with different n_components
n_components_test = 30

# Our implementation
our_pca = PCA(n_components=n_components_test, method='svd')
X_our = our_pca.fit_transform(X_scaled)

# Sklearn implementation
sklearn_pca = SklearnPCA(n_components=n_components_test, svd_solver='full')
X_sklearn = sklearn_pca.fit_transform(X_scaled)

print("Implementation Comparison:")
print("="*60)
print(f"\nNumber of components: {n_components_test}")
print(f"\nTransformed data shape:")
print(f"  Our implementation: {X_our.shape}")
print(f"  Sklearn: {X_sklearn.shape}")

In [ ]:
# Compare explained variance ratios
print("\nExplained Variance Ratios (first 10 components):")
print("-"*50)
print(f"{'Component':<12} {'Our PCA':>15} {'Sklearn':>15} {'Diff':>12}")
print("-"*50)

for i in range(10):
    our_var = our_pca.explained_variance_ratio_[i]
    sklearn_var = sklearn_pca.explained_variance_ratio_[i]
    diff = abs(our_var - sklearn_var)
    print(f"PC{i+1:<10} {our_var:>15.6f} {sklearn_var:>15.6f} {diff:>12.2e}")

print("-"*50)
total_diff = np.sum(np.abs(our_pca.explained_variance_ratio_ - sklearn_pca.explained_variance_ratio_))
print(f"{'Total diff':<12} {'':<15} {'':<15} {total_diff:>12.2e}")

In [ ]:
# Compare principal components (accounting for sign ambiguity)
# Note: Eigenvectors are unique up to sign, so we compare absolute values
print("\nPrincipal Components Comparison:")
print("-"*50)

# For each component, check if they're the same (possibly with sign flip)
for i in range(min(5, n_components_test)):
    our_pc = our_pca.components_[i]
    sklearn_pc = sklearn_pca.components_[i]
    
    # Check correlation (should be +1 or -1 if same component)
    correlation = np.corrcoef(our_pc, sklearn_pc)[0, 1]
    
    # Check if same or sign-flipped
    same_direction = np.allclose(our_pc, sklearn_pc, rtol=1e-5)
    opposite_direction = np.allclose(our_pc, -sklearn_pc, rtol=1e-5)
    
    status = "Match" if same_direction else ("Match (flipped)" if opposite_direction else "Different")
    print(f"PC{i+1}: Correlation = {correlation:.6f}, Status = {status}")

In [ ]:
# Compare transformation results
# Account for possible sign flips in components
sign_corrections = np.sign(np.sum(our_pca.components_ * sklearn_pca.components_, axis=1))
X_our_corrected = X_our * sign_corrections

# Compute differences
transform_diff = np.abs(X_our_corrected - X_sklearn)

print("\nTransformation Comparison (after sign correction):")
print("-"*50)
print(f"Max absolute difference: {np.max(transform_diff):.2e}")
print(f"Mean absolute difference: {np.mean(transform_diff):.2e}")
print(f"Correlation of first component: {np.corrcoef(X_our[:, 0], X_sklearn[:, 0])[0,1]:.6f}")

In [ ]:
# Compare reconstruction error
# Our implementation
X_reconstructed_our = our_pca.inverse_transform(X_our)
mse_our = np.mean((X_scaled - X_reconstructed_our) ** 2)

# Sklearn
X_reconstructed_sklearn = sklearn_pca.inverse_transform(X_sklearn)
mse_sklearn = np.mean((X_scaled - X_reconstructed_sklearn) ** 2)

print("\nReconstruction Error Comparison:")
print("-"*50)
print(f"Our implementation MSE: {mse_our:.6f}")
print(f"Sklearn MSE: {mse_sklearn:.6f}")
print(f"Difference: {abs(mse_our - mse_sklearn):.2e}")

In [ ]:
# Performance comparison
import time

n_trials = 10
our_times = []
sklearn_times = []

for _ in range(n_trials):
    # Our implementation
    start = time.time()
    pca_temp = PCA(n_components=30, method='svd')
    _ = pca_temp.fit_transform(X_scaled)
    our_times.append(time.time() - start)
    
    # Sklearn
    start = time.time()
    pca_temp = SklearnPCA(n_components=30, svd_solver='full')
    _ = pca_temp.fit_transform(X_scaled)
    sklearn_times.append(time.time() - start)

print("\nPerformance Comparison (30 components, averaged over 10 trials):")
print("-"*50)
print(f"Our implementation: {np.mean(our_times)*1000:.2f} +/- {np.std(our_times)*1000:.2f} ms")
print(f"Sklearn: {np.mean(sklearn_times)*1000:.2f} +/- {np.std(sklearn_times)*1000:.2f} ms")
print(f"Ratio (ours/sklearn): {np.mean(our_times)/np.mean(sklearn_times):.2f}x")

In [ ]:
# Visual comparison of projections
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Our implementation
pca_2d_our = PCA(n_components=2, method='svd')
X_2d_our = pca_2d_our.fit_transform(X_scaled)

scatter1 = axes[0].scatter(X_2d_our[:, 0], X_2d_our[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
axes[0].set_xlabel(f'PC1 ({pca_2d_our.explained_variance_ratio_[0]*100:.2f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d_our.explained_variance_ratio_[1]*100:.2f}%)')
axes[0].set_title('Our Implementation')

# Sklearn
pca_2d_sklearn = SklearnPCA(n_components=2)
X_2d_sklearn = pca_2d_sklearn.fit_transform(X_scaled)

scatter2 = axes[1].scatter(X_2d_sklearn[:, 0], X_2d_sklearn[:, 1], c=y, cmap='tab10', s=20, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_2d_sklearn.explained_variance_ratio_[0]*100:.2f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d_sklearn.explained_variance_ratio_[1]*100:.2f}%)')
axes[1].set_title('Sklearn Implementation')

for ax in axes:
    plt.colorbar(scatter1, ax=ax, label='Digit')

plt.suptitle('2D Projection Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Summary & Key Takeaways

### What We Learned

1. **Mathematical Foundation**
   - PCA finds orthogonal directions of maximum variance
   - Two approaches: eigendecomposition and SVD (SVD preferred for stability)
   - Eigenvalues represent variance along each principal component

2. **Implementation Details**
   - Center data before computing components
   - SVD is more numerically stable than eigendecomposition
   - Sign of components is arbitrary (eigenvector uniqueness)

3. **Evaluation Metrics**
   - Explained variance ratio shows information preserved
   - Cumulative variance helps choose n_components
   - Reconstruction error quantifies information loss

4. **Practical Considerations**
   - **Always standardize** features before PCA
   - PCA is **linear** - cannot capture non-linear relationships
   - Principal components lose interpretability
   - Trade-off between dimensionality reduction and information preservation

### Best Practices

1. Standardize features before applying PCA
2. Use scree plot or cumulative variance to choose n_components
3. Start with 90-95% variance threshold for downstream ML tasks
4. Verify reconstruction quality for your specific application
5. Consider non-linear alternatives if PCA projections look inadequate

### Next Steps

- Explore **Kernel PCA** for non-linear dimensionality reduction
- Try **Incremental PCA** for large datasets that don't fit in memory
- Compare with **t-SNE** and **UMAP** for visualization
- Apply PCA as preprocessing for classification/regression tasks